In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import collect_list, col, lit

In [2]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 21:26:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read\
.option("header", "true")\
.format("csv")\
.load("candidate-skills.txt")

In [4]:
df.show()

+------------+----------+
|candidate_id|     skill|
+------------+----------+
|         123|    Python|
|         123|   Tableau|
|         123|PostgreSQL|
|         234|         R|
|         234|   PowerBI|
|         234|SQL Server|
|         345|    Python|
|         345|   Tableau|
+------------+----------+



In [5]:
df.createOrReplaceTempView("candidates")

In [6]:
df.groupBy("candidate_id").agg(collect_list("skill").alias("skills"))\
.filter(col("skills")==lit(["Python", "Tableau", "PostgreSQL"]))\
.select("candidate_id")\
.show()

+------------+
|candidate_id|
+------------+
|         123|
+------------+



In [7]:
spark.sql(
    """
    select candidate_id from candidates
    where skill in ('Python', 'Tableau', 'PostgreSQL')
    group by candidate_id
    having count(skill)=3
    """
).show()

+------------+
|candidate_id|
+------------+
|         123|
+------------+

